In [6]:
import pandas as pd
import numpy as np

In [7]:
# Loading cleaned wc_matches dataset
matches = pd.read_csv('../data/wc_matches.csv')

#formatting the date to proper format
matches['date'] = pd.to_datetime(matches['date'])

print(matches.shape)
print(matches.head(5))

(1298, 10)
        date      home_team    away_team  home_score  away_score  \
0 2014-01-07          Qatar       Jordan         2.0         0.0   
1 2014-01-29         Mexico  South Korea         4.0         0.0   
2 2014-02-01  United States  South Korea         2.0         0.0   
3 2014-03-05      Australia      Ecuador         3.0         4.0   
4 2014-03-05        Austria      Uruguay         1.0         1.0   

          tournament         city        country  neutral  is_friendly  
0  WAFF Championship         Doha          Qatar    False        False  
1           Friendly  San Antonio  United States     True         True  
2           Friendly       Carson  United States    False         True  
3           Friendly       London        England     True         True  
4           Friendly   Klagenfurt        Austria    False         True  


In [11]:
def get_team_stats(team, matches, last_n_years = 6):
    """
    Calculating stats for one team from recent matches
    So, cutoff date means:
    if max or the latest date is 2026-01-01
    and last_n_years = 6
    so subtracting 6 years from the latest year
    we will get 2020-01-01
    so only matches after 2020 are considered
    To keep up with the latest data we are using recent history only.
    """
    cutoff = matches['date'].max() - pd.DateOffset(years=last_n_years)

    #getting matches where the team has played home or away

    #here we are creating a df as home which consists of that team vs all other teams it has played against
    # Such as:
    #If the team = "Brazil"
    #then the home_team away_team  home_score  away_score
    #         Brazil    Argentina    2             0
    #         Brazil    France   2             0
    home = matches[(matches['home_team'] == team) & (matches['date'] >= cutoff)].copy()

    #getting all the away matches for that team only when that team has played as away_team
    #home_team away_team  home_score  away_score
    #Argentina  Brazil      2             0
    #France    Brazil   2             0
    away = matches[(matches['away_team']==team) & (matches['date']>=cutoff)].copy()

    # Calculating Home Statistics
    home_goals_scored = home['home_score'].mean()
    home_goals_conceded = home['away_score'].mean()
    home_wins = (home['home_score'] > home['away_score']).mean()

    # Calculating Away Statistics
    away_goal_scored = away['away_score'].mean()
    away_goal_conceded = away['home_score'].mean()
    away_wins = (away['away_score'] > away['home_score']).mean()

    # Total game
    total_games = len(home) + len(away)
    all_goals_scored = list(home['home_score']) + list(away['away_score'])
    all_goals_conceded = list(home['away_score']) + list(away['home_score'])

    all_goals_scored = pd.Series(all_goals_scored).dropna()
    all_goals_conceded = pd.Series(all_goals_conceded).dropna()

    # Counting total wins
    wins = (
        (home['home_score'] > home['away_score']).sum() +
        (away['away_score'] > away['home_score']).sum()
    )

    draws = (
        (home['home_score'] == home['away_score']).sum() +
        (away['away_score'] == away['home_score']).sum()
    )

    return {
        'team': team,
        'total_games' : total_games,
        'avg_goal_scored' : round(np.mean(all_goals_scored), 3),
        'avg_goal_conceded': round(np.mean(all_goals_conceded), 3),
        'win_rate': round(wins / total_games, 3) if total_games > 0 else 0,
        'draw_rate': round(draws / total_games, 3) if total_games > 0 else 0,
        'home_goals_scored' : round(home_goals_scored, 3),
        'home_goals_conceded' : round(home_goals_conceded, 3),
        'home_win_rate': round(home_wins, 3),
        'away_goals_scored': round(away_goal_scored, 3),
        'away_goals_conceded': round(away_goal_conceded, 3),
        'away_win_rate': round(away_wins, 3)
    }

#testing on a single team
print(get_team_stats("Brazil", matches))

{'team': 'Brazil', 'total_games': 47, 'avg_goal_scored': np.float64(1.727), 'avg_goal_conceded': np.float64(0.955), 'win_rate': np.float64(0.468), 'draw_rate': np.float64(0.255), 'home_goals_scored': np.float64(1.909), 'home_goals_conceded': np.float64(0.818), 'home_win_rate': np.float64(0.583), 'away_goals_scored': np.float64(1.545), 'away_goals_conceded': np.float64(1.091), 'away_win_rate': np.float64(0.348)}
